In [1]:
import numpy as np 
from vosk import KaldiRecognizer , Model
import wave 
import librosa 
import soundfile as sf 
from IPython.display import Audio , display
from scipy.io.wavfile import write
import json

In [2]:
model_path = "vosk-model-small-en-us-0.15"
vosk_small = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-small-en-us-0.15/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from vosk-model-small-en-us-0.15/graph/HCLr.fst vosk-model-small-en-us-0.15/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-small-en-us-0.15/graph/phones/word_boundary.int


In [3]:
model_path = "vosk-model-en-us-0.22"
vosk_us = Model(model_path)

LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=13 max-active=7000 lattice-beam=6
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:11:12:13:14:15
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-en-us-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:279) Loading HCLG from vosk-model-en-us-0.22/graph/HCLG.fst
LOG (VoskAPI:ReadDataFiles():model.cc:297) Loading words from vosk-model-en-us-0.22/graph/words.txt
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-en-us-0.22/graph/phones/word_boundary.int
LOG (VoskAPI:ReadDataFiles():model.cc:315) Loading subtract 

In [4]:
def audio_test(audio_path,recognizer):
    noisy_audio,sampling_rate = librosa.load(audio_path, sr= 16000, mono = True)
    raw_noisy_audio = (noisy_audio*32767).astype(np.int16).tobytes()
    recognizer.AcceptWaveform(raw_noisy_audio)
    final_result = json.loads(recognizer.FinalResult())
    full_transcript = final_result.get("text","")
    return full_transcript
    

In [5]:
us_recognizer = KaldiRecognizer(vosk_us,16000)
us_recognizer.SetWords(True)

In [6]:
small_recognizer = KaldiRecognizer(vosk_small,16000)
small_recognizer.SetWords(True)

In [7]:
test_audio = 'test_audio_files/Standard recording20.wav'

In [8]:
print(audio_test(test_audio,us_recognizer))
display(Audio(test_audio))

i'm in malaysia now and i'm testing the model against trail noise


In [9]:
print(audio_test(test_audio,small_recognizer))
display(Audio(test_audio))

i'm in college now and i'm testing the moto against trail noise


In [10]:
import unicodedata 

def remove_accents(text):
    nfkd_form = unicodedata.normalize('NFKD', text)
    return "".join([c for c in nfkd_form if not unicodedata.combining(c)])

In [11]:
## text preprocessing

def text_preprocessing(text):
    text = text.lower().strip()
    text = contractions.fix(text)
    text = remove_accents(text)
    def convert_decimal(match):
        number = match.group(0)
        integer, decimal = number.split(".")
        integer_words = num2words(int(integer))
        decimal_words = num2words(int(decimal))
        return f"{integer_words} point {decimal_words}"
        
    text = re.sub(r"\b\d+\.\d+\b", convert_decimal, text)
    text = re.sub(r"\b\d+\b", lambda x: num2words(int(x.group(0))), text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.split()


In [12]:
## sequance matching 

def sequance_matching_score(target_tokens, vosk_tokens):

    matcher = difflib.SequenceMatcher(None, target_tokens, vosk_tokens)
    
    report = []
    correct_words_count = 0
    total_teacher_words = len(target_tokens)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        
        if tag == 'equal':
            chunk_len = i2 - i1
            correct_words_count += chunk_len
            
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "match"
                })

        elif tag == 'delete':
            for i in range(i1, i2):
                report.append({
                    "word": target_tokens[i],
                    "status": "Missed"
                })

        elif tag == 'replace':

            teacher_chunk = target_tokens[i1:i2]
            vosk_chunk = vosk_tokens[j1:j2]
            
            for t_word, v_word in zip_longest(teacher_chunk, vosk_chunk, fillvalue=None):
                
                if t_word is None:
                    break  

                if v_word is None:
                    report.append({
                        "word": t_word,
                        "status": "Missed"
                    })
                    continue 

                else:
                    similarity = difflib.SequenceMatcher(None, t_word, v_word).ratio()
                    
                    if similarity > 0.8:
                        correct_words_count += 1
                        report.append({
                            "word": t_word, 
                            "status": f"Accepted typo ({int(similarity*100)}%)"
                        })
                    else:
                        report.append({
                            "word": t_word, 
                            "status": f"Wrong word. Heard '{v_word}'"
                        })

    if total_teacher_words == 0:
        final_score = 0
    else:
        final_score = int((correct_words_count / total_teacher_words) * 100)

    return final_score, report


In [13]:
import difflib
from itertools import zip_longest
import re
import contractions
from num2words import num2words

In [14]:
target = "I'm in college now, and I'm testing the model against real noise."

target_norm = text_preprocessing(target)
vosk_us_norm = text_preprocessing(audio_test(test_audio,us_recognizer))
vosk_small_norm = text_preprocessing(audio_test(test_audio,small_recognizer))
print(target_norm)
print(vosk_us_norm)
print(vosk_small_norm)

['i', 'am', 'in', 'college', 'now', 'and', 'i', 'am', 'testing', 'the', 'model', 'against', 'real', 'noise']
['i', 'am', 'in', 'malaysia', 'now', 'and', 'i', 'am', 'testing', 'the', 'model', 'against', 'trail', 'noise']
['i', 'am', 'in', 'college', 'now', 'and', 'i', 'am', 'testing', 'the', 'moto', 'against', 'trail', 'noise']


In [15]:
final_score1 , report1 = sequance_matching_score(target_norm, vosk_us_norm)
report1

[{'word': 'i', 'status': 'match'},
 {'word': 'am', 'status': 'match'},
 {'word': 'in', 'status': 'match'},
 {'word': 'college', 'status': "Wrong word. Heard 'malaysia'"},
 {'word': 'now', 'status': 'match'},
 {'word': 'and', 'status': 'match'},
 {'word': 'i', 'status': 'match'},
 {'word': 'am', 'status': 'match'},
 {'word': 'testing', 'status': 'match'},
 {'word': 'the', 'status': 'match'},
 {'word': 'model', 'status': 'match'},
 {'word': 'against', 'status': 'match'},
 {'word': 'real', 'status': "Wrong word. Heard 'trail'"},
 {'word': 'noise', 'status': 'match'}]

In [16]:
final_score1

85

In [17]:
final_score2 , report2 = sequance_matching_score(target_norm, vosk_small_norm)
report2

[{'word': 'i', 'status': 'match'},
 {'word': 'am', 'status': 'match'},
 {'word': 'in', 'status': 'match'},
 {'word': 'college', 'status': 'match'},
 {'word': 'now', 'status': 'match'},
 {'word': 'and', 'status': 'match'},
 {'word': 'i', 'status': 'match'},
 {'word': 'am', 'status': 'match'},
 {'word': 'testing', 'status': 'match'},
 {'word': 'the', 'status': 'match'},
 {'word': 'model', 'status': "Wrong word. Heard 'moto'"},
 {'word': 'against', 'status': 'match'},
 {'word': 'real', 'status': "Wrong word. Heard 'trail'"},
 {'word': 'noise', 'status': 'match'}]

In [18]:
final_score2

85

In [19]:
test2 = "test_audio_files/Standard recording 7.wav"
target ="Let's go to the café for a croissant."
target_norm = text_preprocessing(target)
vosk_us_norm = text_preprocessing(audio_test(test2,us_recognizer))
vosk_small_norm = text_preprocessing(audio_test(test2,small_recognizer))
print(target_norm)
print(vosk_us_norm)
print(vosk_small_norm)

print("\n\n")

final_score1 , report1 = sequance_matching_score(target_norm, vosk_us_norm)
final_score2 , report2 = sequance_matching_score(target_norm, vosk_small_norm)

for item in report1:
    print(f"{item['word']:<15} | {item['status']}")
print("_____________________________________")
for item in report2:
    print(f"{item['word']:<15} | {item['status']}")
print("_____________________________________")
print(final_score1,final_score2)

['let', 'us', 'go', 'to', 'the', 'cafe', 'for', 'a', 'croissant']
['let', 'us', 'go', 'to', 'the', 'cafe', 'for', 'a', 'croissant']
['let', 'us', 'go', 'to', 'the', 'cafe', 'for', 'a', 'croissant']



let             | match
us              | match
go              | match
to              | match
the             | match
cafe            | match
for             | match
a               | match
croissant       | match
_____________________________________
let             | match
us              | match
go              | match
to              | match
the             | match
cafe            | match
for             | match
a               | match
croissant       | match
_____________________________________
100 100


## test on mayan's recordings 

In [20]:
def check_fuzzy_keywords(child_speech, keywords, threshold=0.80):
    missing_words = []
    for target in keywords:
        found = False

  
        if target in child_speech:
            found = True
        else:
            for word in child_speech:
                similarity = difflib.SequenceMatcher(None, target, word).ratio()
                if similarity >= threshold:
                    found = True
                    break
        
        if not found:
            missing_words.append(target)

    if len(missing_words) == 0:
        return True
    else:
        return False
    


In [ ]:
from glob import glob
test_data = glob('final_test_audios/mayan/Standard recording *.wav')
for i in test_data:
    display(Audio(i))
   # target_norm = text_preprocessing(target)
    vosk_us_norm = text_preprocessing(audio_test(i,us_recognizer))
    vosk_small_norm = text_preprocessing(audio_test(i,small_recognizer))
    #print(f"target: ")
    print(f"vosk_us: {vosk_us_norm}")
    print(f"vosk_small: {vosk_small_norm}")

    print("\n\n")

In [22]:
targets = [
    "red car",
    "One plus one is two.",
    "a square",             
    "1, 2, 3, 4, 5.",
    "This is my mom.",
    "I love my mom.",
    "Good morning. Good night.",
    "Sunday, Monday, Tuesday, Wednesday, Thursday, Friday.",
    "Under chair.",
    "love family.",
    "dog.",
    "Red.",
]

In [23]:
vosk_us_acc_shadwing= []
vosk_us_acc_fuzzy= []

vosk_small_acc_shadwing= []
vosk_small_acc_fuzzy= []

for i, target in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    vosk_us_norm = text_preprocessing(audio_test(i, us_recognizer))
    vosk_small_norm = text_preprocessing(audio_test(i, small_recognizer))
    
    print(f"target:{target_norm} ")
    print(f"vosk_us: {vosk_us_norm}")
    print(f"vosk_small: {vosk_small_norm}")
    print("\n\n")
    print('fuzzy match')
    passed_us = check_fuzzy_keywords(vosk_us_norm, target_norm)
    passed_small = check_fuzzy_keywords(vosk_small_norm, target_norm)

    vosk_us_acc_fuzzy.append(passed_us)
    vosk_small_acc_fuzzy.append(passed_small)
    
    print(f"vosk_us: {passed_us}")
    print(f"vosk_small: {passed_small}")
    print("\n")
    print("_____________________________________")
    print("\n")
    print("shadowing match")
    final_score1, report1 = sequance_matching_score(target_norm, vosk_us_norm)
    final_score2, report2 = sequance_matching_score(target_norm, vosk_small_norm)

    vosk_us_acc_shadwing.append(final_score1)
    vosk_small_acc_shadwing.append(final_score2)

    for item in report1:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    for item in report2:
        print(f"{item['word']:<15} | {item['status']}")
    print("_____________________________________")
    print(final_score1, final_score2)
    
    print("\n\n")
    print("__________________________________________________________________")

# Fuzzy keyword accuracy (%)
fuzzy_us_accuracy = sum(vosk_us_acc_fuzzy) / len(vosk_us_acc_fuzzy) * 100
fuzzy_small_accuracy = sum(vosk_small_acc_fuzzy) / len(vosk_small_acc_fuzzy) * 100

# Shadowing average score (%)

threshold = 70
shadow_us_pass = [score >= threshold for score in vosk_us_acc_shadwing]
shadow_small_pass = [score >= threshold for score in vosk_small_acc_shadwing]

shadow_us_accuracy = sum(shadow_us_pass) / len(shadow_us_pass) * 100
shadow_small_accuracy = sum(shadow_small_pass) / len(shadow_small_pass) * 100

print(f"Fuzzy Accuracy - US model: {fuzzy_us_accuracy:.2f}%")
print(f"Fuzzy Accuracy - Small model: {fuzzy_small_accuracy:.2f}%")
print(f"Shadowing Accuracy - US model: {shadow_us_accuracy:.2f}%")
print(f"Shadowing Accuracy - Small model: {shadow_small_accuracy:.2f}%")

shadow_us_average = sum(vosk_us_acc_shadwing) / len(vosk_us_acc_shadwing)
shadow_small_average = sum(vosk_small_acc_shadwing) / len(vosk_small_acc_shadwing)

print(f"Shadowing Average Score - US model: {shadow_us_average:.2f}%")
print(f"Shadowing Average Score - Small model: {shadow_small_average:.2f}%")


target:['red', 'car'] 
vosk_us: ['red', 'car']
vosk_small: ['red', 'car']



fuzzy match
vosk_us: True
vosk_small: True


_____________________________________


shadowing match
red             | match
car             | match
_____________________________________
red             | match
car             | match
_____________________________________
100 100



__________________________________________________________________
target:['one', 'plus', 'one', 'is', 'two'] 
vosk_us: ['one', 'last', 'one', 'is', 'two']
vosk_small: ['one', 'last', 'one', 'if', 'to']



fuzzy match
vosk_us: False
vosk_small: False


_____________________________________


shadowing match
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | match
two             | match
_____________________________________
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | Wrong word. Heard 'if'
two             | Wro

In [24]:
targets = [
    ("red car",'fuzzy'),
   ( "One plus one is two.", 'shadow'),
    ("a square",'fuzzy'),             
   ( "1, 2, 3, 4, 5.", 'shadow'),
   ( "This is my mom.", 'shadow'),
   ( "I love my mom.", 'shadow'),
    ("Good morning. Good night.", 'shadow'),
    ("Sunday, Monday, Tuesday, Wednesday, Thursday, Friday." ,'fuzzy'),
   ( "Under chair." ,'fuzzy'),
    ("love family." ,'fuzzy'),
    ("dog." ,'fuzzy'),
    ("Red." ,'fuzzy'),
]

In [25]:
vosk_us_acc_shadwing= []
vosk_us_acc_fuzzy= []

vosk_small_acc_shadwing= []
vosk_small_acc_fuzzy= []

for i, (target, ttype) in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    vosk_us_norm = text_preprocessing(audio_test(i, us_recognizer))
    vosk_small_norm = text_preprocessing(audio_test(i, small_recognizer))
    
    print(f"target:{target_norm} ")
    print(f"vosk_us: {vosk_us_norm}")
    print(f"vosk_small: {vosk_small_norm}")
    print("\n\n")
    if ttype == "fuzzy":
        print('fuzzy match')
        passed_us = check_fuzzy_keywords(vosk_us_norm, target_norm)
        passed_small = check_fuzzy_keywords(vosk_small_norm, target_norm)

        vosk_us_acc_fuzzy.append(passed_us)
        vosk_small_acc_fuzzy.append(passed_small)
    
        print(f"vosk_us: {passed_us}")
        print(f"vosk_small: {passed_small}")

    elif ttype == "shadow":
        print("shadowing match")
        final_score1, report1 = sequance_matching_score(target_norm, vosk_us_norm)
        final_score2, report2 = sequance_matching_score(target_norm, vosk_small_norm)

        vosk_us_acc_shadwing.append(final_score1)
        vosk_small_acc_shadwing.append(final_score2)

        for item in report1:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
        for item in report2:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
        print(final_score1, final_score2)
    
    print("\n\n")
    print("__________________________________________________________________")

# Fuzzy keyword accuracy (%)
fuzzy_us_accuracy = sum(vosk_us_acc_fuzzy) / len(vosk_us_acc_fuzzy) * 100
fuzzy_small_accuracy = sum(vosk_small_acc_fuzzy) / len(vosk_small_acc_fuzzy) * 100

# Shadowing average score (%)

threshold = 70
shadow_us_pass = [score >= threshold for score in vosk_us_acc_shadwing]
shadow_small_pass = [score >= threshold for score in vosk_small_acc_shadwing]

shadow_us_accuracy = sum(shadow_us_pass) / len(shadow_us_pass) * 100
shadow_small_accuracy = sum(shadow_small_pass) / len(shadow_small_pass) * 100

print(f"Fuzzy Accuracy - US model: {fuzzy_us_accuracy:.2f}%")
print(f"Fuzzy Accuracy - Small model: {fuzzy_small_accuracy:.2f}%")
print(f"Shadowing Accuracy - US model: {shadow_us_accuracy:.2f}%")
print(f"Shadowing Accuracy - Small model: {shadow_small_accuracy:.2f}%")

shadow_us_average = sum(vosk_us_acc_shadwing) / len(vosk_us_acc_shadwing)
shadow_small_average = sum(vosk_small_acc_shadwing) / len(vosk_small_acc_shadwing)

print(f"Shadowing Average Score - US model: {shadow_us_average:.2f}%")
print(f"Shadowing Average Score - Small model: {shadow_small_average:.2f}%")


target:['red', 'car'] 
vosk_us: ['red', 'car']
vosk_small: ['red', 'car']



fuzzy match
vosk_us: True
vosk_small: True



__________________________________________________________________
target:['one', 'plus', 'one', 'is', 'two'] 
vosk_us: ['one', 'last', 'one', 'is', 'two']
vosk_small: ['one', 'last', 'one', 'if', 'to']



shadowing match
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | match
two             | match
_____________________________________
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | Wrong word. Heard 'if'
two             | Wrong word. Heard 'to'
_____________________________________
80 40



__________________________________________________________________
target:['a', 'square'] 
vosk_us: ['asquith']
vosk_small: ['as', 'queer']



fuzzy match
vosk_us: False
vosk_small: False



__________________________________________________________________


## Fuzzy vs Shadowing 

| Metric                      | US Model | Small Model |
|----------------------------|----------|-------------|
| **Fuzzy Accuracy**         | 66.67%   | 75.00%      |
| **Shadowing Accuracy**     | 75.00%   | 66.67%      |
| **Shadowing Avg. Score**   | 73.33%   | 67.92%      |

---

## Separated Fuzzy & Shadowing Logic

| Metric                      | US Model | Small Model |
|----------------------------|----------|-------------|
| **Fuzzy Accuracy**         | 57.14%   | 71.43%      |
| **Shadowing Accuracy**     | 100.00%  | 80.00%      |
| **Shadowing Avg. Score**   | 96.00%   | 83.00%      |


# the fuzz lib

In [26]:
from thefuzz import fuzz

In [28]:
vosk_us_acc_shadwing= []
vosk_us_acc_fuzzy= []

vosk_small_acc_shadwing= []
vosk_small_acc_fuzzy= []

for i, (target, ttype) in zip(test_data, targets):
    target_norm = text_preprocessing(target)
    vosk_us_norm = text_preprocessing(audio_test(i, us_recognizer))
    vosk_small_norm = text_preprocessing(audio_test(i, small_recognizer))
    
    print(f"target:{target_norm} ")
    print(f"vosk_us: {vosk_us_norm}")
    print(f"vosk_small: {vosk_small_norm}")
    print("\n\n")
    if ttype == "fuzzy":
        print('fuzzy match')
        passed_us = fuzz.partial_token_set_ratio(vosk_us_norm, target_norm)
        passed_small = fuzz.partial_token_set_ratio(vosk_small_norm, target_norm)

        vosk_us_acc_fuzzy.append(passed_us)
        vosk_small_acc_fuzzy.append(passed_small)
    
        print(f"vosk_us: {passed_us}")
        print(f"vosk_small: {passed_small}")

    elif ttype == "shadow":
        print("shadowing match")
        final_score1, report1 = sequance_matching_score(target_norm, vosk_us_norm)
        final_score2, report2 = sequance_matching_score(target_norm, vosk_small_norm)

        vosk_us_acc_shadwing.append(final_score1)
        vosk_small_acc_shadwing.append(final_score2)

        for item in report1:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
        for item in report2:
            print(f"{item['word']:<15} | {item['status']}")
        print("_____________________________________")
        print(final_score1, final_score2)
    
    print("\n\n")
    print("__________________________________________________________________")

# Fuzzy accuracy (%)
threshold = 70

fuzzy_us_pass = [score >= threshold for score in vosk_us_acc_fuzzy]
fuzzy_small_pass = [score >= threshold for score in vosk_small_acc_fuzzy]

fuzzy_us_accuracy = sum(fuzzy_us_pass) / len(fuzzy_us_pass) * 100
fuzzy_small_accuracy = sum(fuzzy_small_pass) / len(fuzzy_small_pass) * 100

# Shadowing average score (%)

shadow_us_pass = [score >= threshold for score in vosk_us_acc_shadwing]
shadow_small_pass = [score >= threshold for score in vosk_small_acc_shadwing]

shadow_us_accuracy = sum(shadow_us_pass) / len(shadow_us_pass) * 100
shadow_small_accuracy = sum(shadow_small_pass) / len(shadow_small_pass) * 100

print(f"Fuzzy Accuracy - US model: {fuzzy_us_accuracy:.2f}%")
print(f"Fuzzy Accuracy - Small model: {fuzzy_small_accuracy:.2f}%")
print(f"Shadowing Accuracy - US model: {shadow_us_accuracy:.2f}%")
print(f"Shadowing Accuracy - Small model: {shadow_small_accuracy:.2f}%")

fuzzy_us_average = sum(vosk_us_acc_fuzzy) / len(vosk_us_acc_fuzzy)
fuzzy_small_average = sum(vosk_small_acc_fuzzy) / len(vosk_small_acc_fuzzy)

shadow_us_average = sum(vosk_us_acc_shadwing) / len(vosk_us_acc_shadwing)
shadow_small_average = sum(vosk_small_acc_shadwing) / len(vosk_small_acc_shadwing)
print("\n\n")
print("__________________________________________________________________")

print(f"Shadowing Average Score - US model: {shadow_us_average:.2f}%")
print(f"Shadowing Average Score - Small model: {shadow_small_average:.2f}%")
print(f"fuzzy Average Score - US model: {fuzzy_us_average:.2f}%")
print(f"fuzzy Average Score - Small model: {fuzzy_small_average:.2f}%")


target:['red', 'car'] 
vosk_us: ['red', 'car']
vosk_small: ['red', 'car']



fuzzy match
vosk_us: 100
vosk_small: 100



__________________________________________________________________
target:['one', 'plus', 'one', 'is', 'two'] 
vosk_us: ['one', 'last', 'one', 'is', 'two']
vosk_small: ['one', 'last', 'one', 'if', 'to']



shadowing match
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | match
two             | match
_____________________________________
one             | match
plus            | Wrong word. Heard 'last'
one             | match
is              | Wrong word. Heard 'if'
two             | Wrong word. Heard 'to'
_____________________________________
80 40



__________________________________________________________________
target:['a', 'square'] 
vosk_us: ['asquith']
vosk_small: ['as', 'queer']



fuzzy match
vosk_us: 67
vosk_small: 71



__________________________________________________________________
target:[

# Fuzzy logic vs Thefuzz lib 
## Fuzzy logic 

| Metric                      | US Model | Small Model |
|----------------------------|----------|-------------|
| **Fuzzy Accuracy**         | 57.14%   | 71.43%      |
| **Shadowing Accuracy**     | 100.00%  | 80.00%      |
| **Shadowing Avg. Score**   | 96.00%   | 83.00%      |


---

## Thefuzz

| Metric                      | US Model | Small Model |
|----------------------------|----------|-------------|
| **Fuzzy Accuracy**         | 57.14%   | 71.43%      |
| **Shadowing Accuracy**     | 100.00%  | 80.00%      |
| **Shadowing Avg. Score**   | 96.00%   | 83.00%      |
| **Fuzzy Avg. Score**       | 80.57%   | 85.43%      |


# no difference